# Simple Landslide Detection Tool (Sentinel-1 + Sentinel-2)

Core detection logic (SAR anomaly + structural
mask + NDVI confirmation + persistence filter + dedup) is from the
validated version — only the AOI, year, and month selection are configurable,
so this can run over Wayanad, Uganda, or anywhere else with a hilly terrain +
vegetation-loss signature.

**How to use it:**
1. Set `AOI` to your area of interest (examples given below for Wayanad and a
   Uganda landslide-prone district).
2. Set `TARGET_YEAR` and `MONTHS_TO_CHECK` — the months you want scanned.
3. Set `WET_MONTHS` — whichever of those months are the rainy season for your
   AOI (used only to pick the more sensitive SAR threshold; doesn't need to be
   contiguous — this matters for Uganda's two-wet-season climate, unlike India's
   single monsoon).
4. Run all cells. Final candidate table appears as a pandas DataFrame and is
   also saved/downloaded as a CSV.

**Important:** this detects *candidate* landslide/debris-flow scars from SAR
backscatter drop + NDVI loss over sloped, non-built, non-water terrain. The tool has been evaluated in Wayanad, Kerala only.

In [1]:
import math
import datetime
import ee

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-.......')  # <- change to your own GEE project if running elsewhere


In [ ]:
################################################################ ============================================================
# CONFIG — everything you're likely to change lives here
# ============================================================

# ---- AREA OF INTEREST ----

# Example: Bududa district, Uganda (Mount Elgon foothills — Uganda's most
# landslide-prone area). Coordinates are approximate — tighten this box using
# the GEE Code Editor's drawing tool or Google Maps before a real run.
UGANDA_BUDUDA = ee.Geometry.Polygon([[
    [34.20, 0.75], [34.60, 0.75],
    [34.60, 1.05], [34.20, 1.05],
]])

AOI = UGANDA_BUDUDA   # <- swap to  (or your own polygon) as needed

# ---- TIME WINDOW ----
TARGET_YEAR = 2025

# Which calendar months (1-12) to scan this year. Doesn't need to be
# contiguous — e.g. Uganda's two wet seasons might mean [3,4,5,9,10,11].
MONTHS_TO_CHECK = [10, 11]

# Which of the months above count as "wet season" for THIS aoi — controls
# which SAR change threshold gets used (wet season = more soil-moisture noise,
# so a slightly less sensitive threshold is used). Does not need to be
# contiguous or match MONTHS_TO_CHECK exactly.
WET_MONTHS = {10, 11}


###########################################################################################
# ---- BASELINE (reference) PERIOD ----
# A "normal" dry-ish reference window used to measure change against. Must
# NOT overlap MONTHS_TO_CHECK. Original Wayanad baseline used Jan-Apr
# (India's dry season). For Uganda's Bududa area, Dec-Feb is typically drier.
BASELINE_START = f'{TARGET_YEAR - 1}-12-01'
BASELINE_END   = f'{TARGET_YEAR}-02-28'

# ---- DETECTION THRESHOLDS (unchanged from validated Wayanad version) ----
SLOPE_MIN_DEG = 10.0

SAR_CHANGE_DB_MONSOON = -3.0   # used when window falls in WET_MONTHS
SAR_CHANGE_DB_DRY     = -4.0   # used otherwise
NDVI_CHANGE_THR       = -0.22

MIN_AREA_M2      = 8000.0
MIN_CONNECTED_PX = 10

DEDUP_BUFFER_M   = 90      # candidate centroid proximity = "same scar"
TIMEOUT_DAYS     = 90      # cap on waiting for S2 confirmation
REACTIVATION_GAP_DAYS = 300  # ~18 months; re-detection past this counts as new

CS_THRESHOLD          = 0.60  # Cloud Score+ clear-pixel threshold
VALID_FRACTION_MIN_DRY     = 0.85
VALID_FRACTION_MIN_MONSOON = 0.55
POST_MONSOON_GRACE_DAYS    = 45   # extra confirmation days past the last wet month

S1_ORBIT_PASSES    = ['ASCENDING', 'DESCENDING']   # verify denser pass over your AOI first (see diagnostic cell)
SPECKLE_RADIUS_M = 30
HEADING_ASCENDING_DEG  = -12.0
HEADING_DESCENDING_DEG = 192.0
COS_LIA_FLOOR = 0.05

EXPORT_STATUSES = {'dual_confirmed', 'sar_only_timeout'}
# 'optically_contradicted' is kept in the registry (not exported) so you can
# inspect likely false-SAR-positives if curious.


In [ ]:
# ============================================================
# global products, so this works over Uganda without modification.
# ============================================================
FABDEM_COLL = (ee.ImageCollection('projects/sat-io/open-datasets/FABDEM')
               .filterBounds(AOI))

# Grab native projection from an actual tile before mosaicking —
# mosaic() collapses per-tile projection info, so this must happen first.
FABDEM_PROJECTION = FABDEM_COLL.first().projection()

DEM = (FABDEM_COLL
       .select('b1')                      # FABDEM's band is 'b1', not 'DEM'
       .mosaic()
       .setDefaultProjection(FABDEM_PROJECTION)
       .rename('DEM')
       .clip(AOI))

slope_img  = ee.Terrain.slope(DEM).clip(AOI)
aspect_img = ee.Terrain.aspect(DEM).clip(AOI)  # deg CW from N, downslope
slope_mask = slope_img.gte(SLOPE_MIN_DEG)

wc_map = (ee.ImageCollection('ESA/WorldCover/v200').first()
          .select('Map').clip(AOI))
lulc_mask = (wc_map.neq(50)).And(wc_map.neq(80)).And(wc_map.neq(90)).And(wc_map.neq(95))

STRUCTURAL_MASK = (slope_mask.And(lulc_mask)
                   .toInt()
                   .reproject(crs='EPSG:4326', scale=30)
                   .rename('structural_mask'))


In [ ]:
# ============================================================
# SAFE COMPOSITES + SENTINEL-2 NDVI (Cloud Score+ masking) — unchanged
# ============================================================
def _masked_constant(band_name):
    return (ee.Image.constant(0).rename(band_name)
            .updateMask(ee.Image.constant(0)))


def safe_median(coll, band_name):
    return ee.Image(ee.Algorithms.If(
        coll.size().gt(0), coll.median(), _masked_constant(band_name)))


S2_SR = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
CSP   = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')


def s2_ndvi(image):
    clean = image.select('cs_cdf').gte(CS_THRESHOLD)
    return ee.Image(image.normalizedDifference(['B8', 'B4'])
                    .rename('NDVI').updateMask(clean)
                    .copyProperties(image, ['system:time_start']))


def s2_ndvi_composite(start, end):
    coll = (S2_SR.filterBounds(AOI).filterDate(start, end)
            .linkCollection(CSP, ['cs_cdf'])
            .map(s2_ndvi))
    return safe_median(coll, 'NDVI').clip(AOI)


def valid_fraction_over(img, geom):
    vf = (img.mask()
          .reduceRegion(ee.Reducer.mean(), geom, scale=20,
                        maxPixels=1e7, tileScale=4)
          .values().get(0))
    return ee.Number(ee.Algorithms.If(vf, vf, 0)).getInfo()


In [ ]:
# ============================================================
# SENTINEL-1 RTC PIPELINE — dual-pass (ASCENDING + DESCENDING)
# ============================================================
def apply_rtc(img):
    pass_dir = ee.String(img.get('orbitProperties_pass'))
    heading = ee.Number(ee.Algorithms.If(
        pass_dir.equals('DESCENDING'),
        HEADING_DESCENDING_DEG, HEADING_ASCENDING_DEG))
    look_azimuth_deg = heading.add(90)

    theta_i = img.select('angle').multiply(math.pi / 180)
    alpha   = slope_img.multiply(math.pi / 180)
    phi_n   = aspect_img.multiply(math.pi / 180)
    phi_r   = ee.Image.constant(look_azimuth_deg).multiply(math.pi / 180)
    phi_rel = phi_n.subtract(phi_r)

    cos_lia = (theta_i.cos().multiply(alpha.cos())
               .subtract(theta_i.sin().multiply(alpha.sin())
                         .multiply(phi_rel.cos())))

    alpha_r = alpha.tan().multiply(phi_rel.cos()).multiply(-1).atan()
    layover = alpha_r.gte(theta_i)
    shadow  = alpha_r.lte(theta_i.subtract(math.pi / 2))
    geom_mask = layover.Or(shadow).Not()

    cos_lia_safe = cos_lia.max(COS_LIA_FLOOR)
    gamma0 = (img.select('VH')
              .multiply(theta_i.cos()).divide(cos_lia_safe)
              .updateMask(geom_mask))
    return ee.Image(gamma0.rename('VH_rtc')
                    .copyProperties(img, ['system:time_start',
                                          'orbitProperties_pass']))


def get_s1_collection_for_pass(orbit_pass, start, end):
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(AOI)
            .filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains(
                'transmitterReceiverPolarisation', 'VH'))
            .filter(ee.Filter.eq('orbitProperties_pass', orbit_pass))
            .map(apply_rtc))


def build_composite_both_passes(start, end, band='VH_rtc', reducer='median'):
    """
    Builds one composite per pass, then mosaics them together so the
    AOI is covered by whichever pass has data at each pixel.
    Falls back gracefully if a pass has zero scenes in this window.
    """
    per_pass_images = []
    for orbit_pass in S1_ORBIT_PASSES:
        coll = get_s1_collection_for_pass(orbit_pass, start, end).select(band)
        count = coll.size().getInfo()
        if count == 0:
            print(f"  [skip] {orbit_pass}: 0 scenes in {start}\u2013{end}")
            continue
        img = coll.median() if reducer == 'median' else coll.mean()
        per_pass_images.append(img.set('orbit_pass_used', orbit_pass))

    if not per_pass_images:
        raise ValueError(f"No S1 scenes on any pass for window {start}\u2013{end}")

    # mosaic() fills gaps pass-by-pass; last image in list wins on overlap
    return ee.ImageCollection(per_pass_images).mosaic()


def sar_change_both_passes(baseline_start, baseline_end, month_start, month_end, band='VH_rtc'):
    baseline_img = build_composite_both_passes(baseline_start, baseline_end, band)
    monitor_img  = build_composite_both_passes(month_start, month_end, band)
    return monitor_img.subtract(baseline_img).rename('sar_change_db')


def s1_vh_composite(start, end):
    composite = build_composite_both_passes(start, end, band='VH_rtc')
    return (composite.focal_median(SPECKLE_RADIUS_M, 'circle', 'meters')
            .clip(AOI))


In [ ]:
# ============================================================
# BASELINE COVERAGE-GAP DIAGNOSTIC
# Flags any part of the AOI where a monitoring-window pass has data
# but the baseline has NO scenes on that same pass — those pixels get
# masked out (no-data) in the anomaly diff rather than silently
# defaulting to 0/no-change. This cell just reports the gap size so
# you know if it's negligible or worth backfilling baseline scenes for.
# ============================================================
def pass_has_baseline_coverage(orbit_pass):
    coll = get_s1_collection_for_pass(orbit_pass, BASELINE_START, BASELINE_END)
    return coll.size().getInfo() > 0


def pass_footprint(orbit_pass, start, end):
    """Union footprint of all scenes for one pass in a window, or None if empty."""
    coll = get_s1_collection_for_pass(orbit_pass, start, end)
    if coll.size().getInfo() == 0:
        return None
    return coll.geometry().dissolve(maxError=30)


print("Checking baseline coverage per pass...")
passes_without_baseline = []
for orbit_pass in S1_ORBIT_PASSES:
    has_cov = pass_has_baseline_coverage(orbit_pass)
    print(f"  {orbit_pass}: baseline coverage = {has_cov}")
    if not has_cov:
        passes_without_baseline.append(orbit_pass)

if not passes_without_baseline:
    print("\nAll passes have baseline coverage. No gap to check.")
else:
    print(f"\n{passes_without_baseline} have NO baseline scenes.")
    print("Checking how much unique ground this affects using your first sweep window...")

    first_period = periods[0] if 'periods' in dir() else None
    if first_period is None:
        print("  (Run this after the sweep-windows cell so `periods` exists, "
              "or set check_start/check_end manually below.)")
    else:
        check_start = first_period['start'].isoformat()
        check_end = first_period['end'].isoformat()

        covered_by_baseline_passes = None
        for orbit_pass in S1_ORBIT_PASSES:
            if orbit_pass in passes_without_baseline:
                continue
            fp = pass_footprint(orbit_pass, check_start, check_end)
            if fp is None:
                continue
            covered_by_baseline_passes = (fp if covered_by_baseline_passes is None
                                           else covered_by_baseline_passes.union(fp, maxError=30))

        gap_area_km2 = 0
        for orbit_pass in passes_without_baseline:
            fp = pass_footprint(orbit_pass, check_start, check_end)
            if fp is None:
                continue
            aoi_geom = AOI if isinstance(AOI, ee.Geometry) else AOI.geometry()
            uncovered = fp.intersection(aoi_geom, maxError=30)
            if covered_by_baseline_passes is not None:
                uncovered = uncovered.difference(covered_by_baseline_passes, maxError=30)
            area_m2 = uncovered.area(maxError=30).getInfo()
            gap_area_km2 += area_m2 / 1e6
            print(f"  {orbit_pass}-only gap over AOI: {area_m2 / 1e6:.2f} km\u00b2")

        print(f"\nTotal blind-spot area (no baseline to compare against): "
              f"{gap_area_km2:.2f} km\u00b2")
        print("These pixels will be masked out (skipped) in every SAR anomaly "
              "check for the whole run \u2014 landslides here would not be "
              "detected.")


In [ ]:
# ============================================================
# SAR ANOMALY MASK + DEDUP HELPERS — unchanged
# ============================================================
def compute_sar_anomaly_mask(start_str, end_str, is_wet, year):
    """SAR anomaly mask for one window, compared against that window's OWN
    year's baseline."""
    sar_thr = SAR_CHANGE_DB_MONSOON if is_wet else SAR_CHANGE_DB_DRY
    sar_now = s1_vh_composite(start_str, end_str)
    sar_diff = sar_now.subtract(BASELINE_SAR)
    is_sar = sar_diff.lt(sar_thr).unmask(0)
    return is_sar.And(STRUCTURAL_MASK)


def _haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = (math.sin(dphi / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))


def find_nearby_candidate(registry, lat, lon, buffer_m, detect_date,
                           max_gap_days=REACTIVATION_GAP_DAYS):
    """Return the matching candidate only if it's spatially AND temporally
    close. Spatially-close but temporally-distant hits are treated as
    reactivations, not duplicates, and return None."""
    for cand in registry:
        d = _haversine_m(lat, lon, cand['centroid_lat'], cand['centroid_lon'])
        if d > buffer_m:
            continue
        last_seen = cand.get('last_detect_date', cand['sar_detect_date'])
        gap = abs((detect_date - last_seen).days)
        if gap <= max_gap_days:
            return cand
    return None


def anomaly_present(lat, lon, start_str, end_str, is_wet, buffer_m=90):
    """Check if a specific point still shows a SAR anomaly in a different
    (later) window — used for the persistence filter."""
    anomaly = compute_sar_anomaly_mask(start_str, end_str, is_wet, None)
    pt = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
    val = anomaly.reduceRegion(ee.Reducer.max(), pt, scale=30,
                               maxPixels=1e6, bestEffort=True).getInfo()
    return bool(list(val.values())[0])


In [ ]:
# ============================================================
# BUILD SWEEP WINDOWS from MONTHS_TO_CHECK (replaces the old
# known-event-padded sweep range). 16-day windows, 8-day stride.
# ============================================================
def month_bounds(year, month):
    start = datetime.date(year, month, 1)
    if month == 12:
        end = datetime.date(year + 1, 1, 1)
    else:
        end = datetime.date(year, month + 1, 1)
    return start, end


def months_to_ranges(year, months):
    """Merge consecutive selected months into contiguous date ranges so
    windows aren't scanned twice at month boundaries. Non-consecutive
    months (e.g. Uganda's two wet seasons) stay as separate ranges."""
    months = sorted(set(months))
    ranges = []
    run_start = run_end = None
    prev_m = None
    for m in months:
        s, e = month_bounds(year, m)
        if prev_m is not None and m == prev_m + 1:
            run_end = e
        else:
            if run_start is not None:
                ranges.append((run_start, run_end))
            run_start, run_end = s, e
        prev_m = m
    if run_start is not None:
        ranges.append((run_start, run_end))
    return ranges


MERGED_RANGES = months_to_ranges(TARGET_YEAR, MONTHS_TO_CHECK)
print("Sweep ranges for", TARGET_YEAR, ":")
for r in MERGED_RANGES:
    print(f"  {r[0]} to {r[1]}  ({(r[1]-r[0]).days} days)")

periods = []
for range_start, range_end in MERGED_RANGES:
    total_days = (range_end - range_start).days
    for offset in range(0, max(total_days - 16, 0) + 1, 8):
        ws = range_start + datetime.timedelta(days=offset)
        we = ws + datetime.timedelta(days=16)
        mid = ws + datetime.timedelta(days=8)
        periods.append({'start': ws, 'end': we, 'mid': mid})

print(f"\nTotal windows to sweep: {len(periods)}")


In [ ]:
# ============================================================
# BASELINE — one reference composite for the whole run, built from
# BASELINE_START/BASELINE_END. Prints a valid-pixel sanity check —
# if this comes back 0 or very low, widen the baseline window (poor
# orbit coverage in that period for your AOI).
# ============================================================
print(f"Building baseline: {BASELINE_START} to {BASELINE_END}")
BASELINE_SAR = s1_vh_composite(BASELINE_START, BASELINE_END)
BASELINE_NDVI = s2_ndvi_composite(BASELINE_START, BASELINE_END)

n = BASELINE_SAR.reduceRegion(ee.Reducer.count(), AOI, scale=100,
                              maxPixels=1e8, bestEffort=True).getInfo()
print(f"Baseline SAR valid pixel count: {n}")


In [ ]:
# ============================================================
# PHASE 1 — SAR SWEEP + CANDIDATE REGISTRY (unchanged logic)
# ============================================================
print(f"PHASE 1: sweeping {len(periods)} SAR windows ...")
registry = []

for p in periods:
    is_wet = p['start'].month in WET_MONTHS
    anomaly = compute_sar_anomaly_mask(p['start'].isoformat(),
                                       p['end'].isoformat(), is_wet, TARGET_YEAR)
    connected_px = anomaly.connectedPixelCount(maxSize=256, eightConnected=True)
    landslide_mask = anomaly.updateMask(connected_px.gte(MIN_CONNECTED_PX)).selfMask()

    polygons = landslide_mask.reduceToVectors(
        geometry=AOI, scale=30, eightConnected=True,
        maxPixels=1e8, reducer=ee.Reducer.countEvery(),
        labelProperty='cluster_id', bestEffort=True)
    polygons = (polygons
                .map(lambda f: f.set('area_m2', f.geometry().area(maxError=1))
                                 .set('centroid', f.geometry().centroid(maxError=10)
                                                      .coordinates()))
                .filter(ee.Filter.gte('area_m2', MIN_AREA_M2)))

    feats = polygons.getInfo()['features']
    for f in feats:
        lon, lat = f['properties']['centroid']
        area_m2 = f['properties']['area_m2']
        match = find_nearby_candidate(registry, lat, lon, DEDUP_BUFFER_M, p['mid'])
        if match is not None:
            match['last_detect_date'] = p['mid']
            continue
        registry.append({
            'geometry': f['geometry'],
            'centroid_lat': lat,
            'centroid_lon': lon,
            'area_m2': area_m2,
            'sar_window_start': p['start'].isoformat(),
            'sar_window_end': p['end'].isoformat(),
            'sar_detect_date': p['mid'],
            'last_detect_date': p['mid'],
            'status': 'sar_pending',
            's2_confirm_date': None,
        })
    print(f"  window [{p['start']} - {p['end']}]: "
          f"{len(feats)} raw, registry size = {len(registry)}")

print(f"PHASE 1 done: {len(registry)} candidate scars.")

# Final light dedup pass — catches any two candidates that ended up
# just inside each other's buffer across non-adjacent windows.
finalized = []
for cand in registry:
    match = find_nearby_candidate(finalized, cand['centroid_lat'],
                                   cand['centroid_lon'], DEDUP_BUFFER_M,
                                   cand['sar_detect_date'])
    if match is None:
        finalized.append(cand)
    else:
        match['last_detect_date'] = max(match.get('last_detect_date',
                                                    match['sar_detect_date']),
                                         cand['sar_detect_date'])
registry = finalized
print(f"PHASE 1 after final dedup pass: {len(registry)} candidates.")


In [ ]:
# ============================================================
# PHASE 1.5 — PERSISTENCE FILTER (parallelized)
# Re-check each candidate in a LATER, independent window to confirm the
# SAR anomaly isn't a one-off blip. The window-selection logic is unchanged;
# only the GEE round-trip (anomaly_present) is now dispatched in parallel,
# since that's what was making this take 9+ minutes sequentially.
# ============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed

PERSISTENCE_LAG_DAYS = 30
MIN_PERSISTENCE_LAG_DAYS = 10
PERSISTENCE_WINDOW_DAYS = 30
TODAY = datetime.date.today()
PERSISTENCE_MAX_WORKERS = 10   # same pattern as Phase 2 GSI attribution

print(f"\nPHASE 1.5: persistence check on {len(registry)} candidates "
      f"(ideal +{PERSISTENCE_LAG_DAYS}d, min +{MIN_PERSISTENCE_LAG_DAYS}d, "
      f"{PERSISTENCE_WINDOW_DAYS}d window)...")

# ---- Step 1: local-only window selection (no API calls, cheap, sequential) ----
to_check = []
pending = []

for cand in registry:
    orig_end = datetime.date.fromisoformat(cand['sar_window_end'])

    ideal_start = orig_end + datetime.timedelta(days=PERSISTENCE_LAG_DAYS)
    ideal_end = ideal_start + datetime.timedelta(days=PERSISTENCE_WINDOW_DAYS)

    if ideal_end <= TODAY:
        check_start, check_end = ideal_start, ideal_end
    else:
        fallback_end = TODAY
        fallback_start = fallback_end - datetime.timedelta(days=PERSISTENCE_WINDOW_DAYS)
        earliest_allowed_start = orig_end + datetime.timedelta(days=MIN_PERSISTENCE_LAG_DAYS)

        if fallback_start >= earliest_allowed_start:
            check_start, check_end = fallback_start, fallback_end
        else:
            cand['persistence_check_start'] = None
            cand['persistence_check_end'] = None
            cand['status'] = 'too_recent_for_persistence_check'
            cand['persistent'] = None
            pending.append(cand)
            continue

    cand['persistence_check_start'] = check_start.isoformat()
    cand['persistence_check_end'] = check_end.isoformat()
    to_check.append(cand)

# ---- Step 2: the actual GEE round-trips, dispatched in parallel ----
def _check_one(cand):
    check_start = cand['persistence_check_start']
    check_end = cand['persistence_check_end']
    is_wet_check = datetime.date.fromisoformat(check_start).month in WET_MONTHS
    still_there = anomaly_present(
        cand['centroid_lat'], cand['centroid_lon'],
        check_start, check_end,
        is_wet_check, buffer_m=DEDUP_BUFFER_M)
    return cand, still_there

persistent = []
dropped = []
done = 0

with ThreadPoolExecutor(max_workers=PERSISTENCE_MAX_WORKERS) as executor:
    futures = [executor.submit(_check_one, cand) for cand in to_check]
    for fut in as_completed(futures):
        cand, still_there = fut.result()
        cand['persistent'] = still_there
        if still_there:
            cand['status'] = 'sar_confirmed_persistent'
            persistent.append(cand)
        else:
            cand['status'] = 'sar_dropped_transient'
            dropped.append(cand)
        done += 1
        if done % 10 == 0 or done == len(to_check):
            print(f"  ...{done}/{len(to_check)} checked")

print(f"PHASE 1.5 done: {len(persistent)} persistent, "
      f"{len(dropped)} dropped as transient, "
      f"{len(pending)} still too recent even for a fallback check "
      f"(registry {len(registry)} -> {len(persistent) + len(pending)}).")

registry = persistent + pending


In [ ]:
# ============================================================
# PHASE 2 — SLOPE/ASPECT + ASYNC S2 CONFIRMATION (unchanged logic)
# ============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed

S2_COMPOSITE_WINDOW_DAYS = 12
SEARCH_BACKWARD_BUFFER_DAYS = 10

# Last selected month's end — used for the post-wet-season confirmation grace period
_wet_months_in_range = WET_MONTHS if WET_MONTHS else set(MONTHS_TO_CHECK)
_last_wet_month = max(_wet_months_in_range)
_, WET_SEASON_END = month_bounds(TARGET_YEAR, _last_wet_month)
WET_SEASON_END = WET_SEASON_END - datetime.timedelta(days=1)


def is_wet_candidate(cand):
    d = cand['sar_detect_date']
    if isinstance(d, str):
        d = datetime.date.fromisoformat(d)
    return d.month in WET_MONTHS


def get_valid_fraction_min(cand):
    return VALID_FRACTION_MIN_MONSOON if is_wet_candidate(cand) else VALID_FRACTION_MIN_DRY


def process_candidate(cand):
    """Runs entirely inside a worker thread. Wrapped in try/except so one
    candidate's Earth Engine failure doesn't kill the whole batch."""
    try:
        geom = ee.Geometry(cand['geometry'])

        terrain_stats = (slope_img.rename('slope').addBands(aspect_img.rename('aspect'))
                         .reduceRegion(ee.Reducer.mean(), geom, scale=30,
                                       maxPixels=1e6, tileScale=4)
                         .getInfo())
        cand['slope_deg'] = terrain_stats.get('slope')
        cand['aspect_deg'] = terrain_stats.get('aspect')

        sar_date = cand['sar_detect_date']
        window_start = datetime.date.fromisoformat(cand['sar_window_start'])
        today = datetime.date.today()

        search_start = min(window_start, sar_date) - datetime.timedelta(
            days=SEARCH_BACKWARD_BUFFER_DAYS)

        flat_deadline = sar_date + datetime.timedelta(days=TIMEOUT_DAYS)
        if is_wet_candidate(cand):
            post_wet_deadline = WET_SEASON_END + datetime.timedelta(days=POST_MONSOON_GRACE_DAYS)
            timeout_date = min(max(flat_deadline, post_wet_deadline), today)
        else:
            timeout_date = min(flat_deadline, today)

        check_date = search_start
        confirmed = False
        while check_date <= timeout_date:
            w_start = check_date.isoformat()
            w_end = (check_date + datetime.timedelta(days=S2_COMPOSITE_WINDOW_DAYS)).isoformat()

            ndvi_now = s2_ndvi_composite(w_start, w_end)
            vf = valid_fraction_over(ndvi_now, geom)
            if vf >= get_valid_fraction_min(cand):
                ndvi_diff_val = (ndvi_now.subtract(BASELINE_NDVI)
                                .reduceRegion(ee.Reducer.mean(), geom, scale=20,
                                              maxPixels=1e6, tileScale=4)
                                .values().get(0).getInfo())
                if ndvi_diff_val is not None and ndvi_diff_val <= NDVI_CHANGE_THR:
                    cand['status'] = 'dual_confirmed'
                else:
                    cand['status'] = 'optically_contradicted'
                cand['ndvi_diff_observed'] = ndvi_diff_val
                cand['s2_confirm_date'] = check_date.isoformat()
                cand['s2_window'] = f"{w_start} to {w_end}"
                confirmed = True
                break
            check_date += datetime.timedelta(days=8)

        if not confirmed:
            cand['status'] = 'sar_only_timeout'
            cand['ndvi_diff_observed'] = None
        cand['_error'] = None

    except Exception as e:
        cand['status'] = 'error'
        cand['_error'] = str(e)
        cand['ndvi_diff_observed'] = None

    return cand


print(f"PHASE 2: slope/aspect + S2 confirmation loop ({len(registry)} candidates, parallel)...")
results = []
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {executor.submit(process_candidate, cand): cand for cand in registry}
    for i, future in enumerate(as_completed(futures)):
        results.append(future.result())
        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(registry)} done...")

registry = results

print("PHASE 2 done. Status counts:")
for s in ('dual_confirmed', 'sar_only_timeout', 'optically_contradicted', 'error'):
    print(f"  {s}: {sum(1 for c in registry if c['status'] == s)}")

n_errors = sum(1 for c in registry if c['status'] == 'error')
if n_errors:
    print(f"\n{n_errors} candidates failed. Inspect cand['_error'] on those, e.g.:")
    for c in registry:
        if c['status'] == 'error':
            print(f"  centroid ({c['centroid_lat']:.4f},{c['centroid_lon']:.4f}): {c['_error']}")


In [ ]:
# ============================================================
# FINAL OUTPUT — DataFrame (viewable in Colab) + CSV download
# ============================================================
import pandas as pd

COLUMNS = ['sar_detect_date', 'sar_window_start', 'sar_window_end',
          'status', 's2_confirm_date', 'ndvi_diff_observed',
          'area_m2', 'slope_deg', 'aspect_deg',
          'centroid_lat', 'centroid_lon']

rows = []
for cand in registry:
    if cand['status'] not in EXPORT_STATUSES:
        continue
    row = {k: cand.get(k) for k in COLUMNS}
    row['sar_detect_date'] = cand['sar_detect_date'].isoformat()
    rows.append(row)

landslide_df = pd.DataFrame(rows, columns=COLUMNS)
landslide_df = landslide_df.sort_values('sar_detect_date').reset_index(drop=True)

print(f"{len(landslide_df)} candidates exported "
      f"({sum(1 for c in registry if c['status'] == 'optically_contradicted')} "
      f"optically-contradicted candidates excluded but kept in `registry` for inspection).")

display(landslide_df)

OUT_PATH = f'landslide_candidates_{TARGET_YEAR}.csv'
landslide_df.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")

# Download in Colab (no-op / safe to ignore if running elsewhere)
try:
    from google.colab import files
    files.download(OUT_PATH)
except Exception:
    print("Not running in Colab (or download blocked) — CSV is saved locally at", OUT_PATH)
